# Reto 9: Sistema de Gestión de Calificaciones

## Programación para Ciencia de Datos
### Instituto Politécnico Nacional
### Febrero - Julio 2026

In [ ]:
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple, Optional
from datetime import datetime
import json
import os

## PARTE 1: Carga y Exploración de Datos

In [ ]:
def cargar_datos() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Carga los datos de estudiantes, calificaciones y materias.
    Retorna: (df_estudiantes, df_calificaciones, df_materias)
    """
    estudiantes = pd.DataFrame({
        'boleta': ['2021630001', '2021630002', '2021630003', '2021630004', '2021630005',
                   '2022630001', '2022630002', '2022630003', '2022630004', '2022630005',
                   '2023630001', '2023630002', '2023630003', '2023630004', '2023630005'],
        'nombre': ['Juan Pérez García', 'María López Ruiz', 'Pedro Sánchez Torres',
                   'Ana Martínez Díaz', 'Luis Rodríguez Vega', 'Carmen Flores Luna',
                   'Roberto Díaz Mora', 'Laura Torres Silva', 'Diego Ramírez Cruz',
                   'Sofía Vargas Romo', 'Carlos Mendoza Ríos', 'Patricia Ortiz León',
                   'Miguel Ángel Castro', 'Fernanda Reyes Paz', 'Andrés Guzmán Villa'],
        'semestre': [4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2],
        'carrera': ['CD'] * 15,
        'email': ['juan.perez@ipn.mx', 'maria.lopez@ipn.mx', 'pedro.sanchez@ipn.mx',
                  'ana.martinez@ipn.mx', 'luis.rodriguez@ipn.mx', 'carmen.flores@ipn.mx',
                  'roberto.diaz@ipn.mx', 'laura.torres@ipn.mx', 'diego.ramirez@ipn.mx',
                  'sofia.vargas@ipn.mx', 'carlos.mendoza@ipn.mx', 'patricia.ortiz@ipn.mx',
                  'miguel.castro@ipn.mx', 'fernanda.reyes@ipn.mx', 'andres.guzman@ipn.mx']
    })

    materias = pd.DataFrame({
        'materia_id': ['MAT101', 'MAT102', 'PROG101', 'PROG102', 'EST101', 'EST102', 'BD101'],
        'nombre': ['Cálculo Diferencial', 'Cálculo Integral', 'Programación I',
                   'Programación II', 'Probabilidad', 'Estadística Inferencial',
                   'Bases de Datos'],
        'creditos': [8, 8, 6, 6, 6, 6, 6],
        'semestre_materia': [1, 2, 1, 2, 2, 3, 3]
    })

    np.random.seed(42)
    calificaciones_data = []

    for boleta in estudiantes['boleta']:
        semestre = estudiantes[estudiantes['boleta'] == boleta]['semestre'].values[0]
        materias_cursadas = materias[materias['semestre_materia'] <= semestre]['materia_id'].tolist()

        for materia in materias_cursadas:
            base = np.random.uniform(5, 10)
            p1 = round(min(10, max(0, base + np.random.normal(0, 1))), 1)
            p2 = round(min(10, max(0, base + np.random.normal(0, 1))), 1)
            final = round(min(10, max(0, base + np.random.normal(0, 0.5))), 1)

            if np.random.random() < 0.05:
                p2 = np.nan

            calificaciones_data.append({
                'boleta': boleta,
                'materia_id': materia,
                'parcial_1': p1,
                'parcial_2': p2,
                'final': final
            })

    calificaciones = pd.DataFrame(calificaciones_data)
    return estudiantes, calificaciones, materias

In [ ]:
def info_general(df_estudiantes: pd.DataFrame, df_calificaciones: pd.DataFrame) -> Dict:
    """
    Genera información general del sistema.
    """
    resultado = {
        "total_estudiantes": len(df_estudiantes),
        "total_registros_calif": len(df_calificaciones),
        "semestres": sorted(df_estudiantes['semestre'].unique().tolist()),
        "materias_con_registros": df_calificaciones['materia_id'].nunique()
    }
    return resultado

In [ ]:
def validar_datos(df_calificaciones: pd.DataFrame) -> Dict:
    """
    Valida la integridad de los datos.
    """
    cols_calif = ['parcial_1', 'parcial_2', 'final']

    # Filas que tienen al menos un valor nulo
    registros_con_nulos = int(df_calificaciones[cols_calif].isna().any(axis=1).sum())

    # Calificaciones fuera del rango [0, 10]
    fuera_rango = 0
    for col in cols_calif:
        serie = df_calificaciones[col].dropna()
        fuera_rango += int(((serie < 0) | (serie > 10)).sum())

    datos_validos = (registros_con_nulos == 0) and (fuera_rango == 0)

    resultado = {
        "registros_con_nulos": registros_con_nulos,
        "calificaciones_fuera_rango": fuera_rango,
        "datos_validos": datos_validos
    }
    return resultado

## PARTE 2: Consultas y Filtros

In [ ]:
def buscar_estudiante(df_estudiantes: pd.DataFrame, criterio: str, valor: str) -> pd.DataFrame:
    """
    Busca estudiantes por diferentes criterios.
    criterio: 'boleta', 'nombre' (parcial) o 'semestre'
    """
    if criterio == 'nombre':
        mask = df_estudiantes['nombre'].str.contains(valor, case=False, na=False)
        return df_estudiantes[mask].reset_index(drop=True)
    elif criterio == 'boleta':
        return df_estudiantes[df_estudiantes['boleta'] == valor].reset_index(drop=True)
    elif criterio == 'semestre':
        return df_estudiantes[df_estudiantes['semestre'] == int(valor)].reset_index(drop=True)
    else:
        raise ValueError(f"Criterio '{criterio}' no válido. Usa: 'boleta', 'nombre' o 'semestre'.")

In [ ]:
def obtener_kardex(boleta: str, df_estudiantes: pd.DataFrame,
                   df_calificaciones: pd.DataFrame, df_materias: pd.DataFrame) -> Dict:
    """
    Obtiene el kardex completo de un estudiante.
    """
    resultado = {
        "estudiante": None,
        "materias": None,
        "promedio_general": None,
        "creditos_cursados": None,
        "materias_aprobadas": None,
        "materias_reprobadas": None
    }

    # 1. Obtener datos del estudiante
    est_row = df_estudiantes[df_estudiantes['boleta'] == boleta]
    if est_row.empty:
        return resultado

    resultado['estudiante'] = est_row.iloc[0].to_dict()

    # 2. Filtrar calificaciones del estudiante
    calif = df_calificaciones[df_calificaciones['boleta'] == boleta].copy()

    # 3. Unir con nombre de materia y créditos
    calif = calif.merge(df_materias[['materia_id', 'nombre', 'creditos']], on='materia_id', how='left')

    # 4. Calcular promedio por materia: (p1 + p2 + final) / 3
    calif['promedio'] = calif[['parcial_1', 'parcial_2', 'final']].mean(axis=1).round(2)

    # 5. Estatus
    calif['estatus'] = calif['promedio'].apply(lambda x: 'Aprobada' if x >= 6.0 else 'Reprobada')

    cols_mostrar = ['materia_id', 'nombre', 'parcial_1', 'parcial_2', 'final', 'promedio', 'estatus', 'creditos']
    resultado['materias'] = calif[cols_mostrar].reset_index(drop=True)

    resultado['promedio_general'] = round(calif['promedio'].mean(), 2)
    resultado['creditos_cursados'] = int(calif['creditos'].sum())
    resultado['materias_aprobadas'] = int((calif['estatus'] == 'Aprobada').sum())
    resultado['materias_reprobadas'] = int((calif['estatus'] == 'Reprobada').sum())

    return resultado

In [ ]:
def filtrar_por_rendimiento(df_calificaciones: pd.DataFrame,
                            df_estudiantes: pd.DataFrame,
                            min_promedio: float = None,
                            max_promedio: float = None) -> pd.DataFrame:
    """
    Filtra estudiantes por rango de promedio general.
    """
    # Calcular promedio por materia, luego promedio general por estudiante
    df = df_calificaciones.copy()
    df['promedio_materia'] = df[['parcial_1', 'parcial_2', 'final']].mean(axis=1)
    promedio_est = df.groupby('boleta')['promedio_materia'].mean().round(2).reset_index()
    promedio_est.columns = ['boleta', 'promedio_general']

    # Aplicar filtros
    if min_promedio is not None:
        promedio_est = promedio_est[promedio_est['promedio_general'] >= min_promedio]
    if max_promedio is not None:
        promedio_est = promedio_est[promedio_est['promedio_general'] <= max_promedio]

    # Unir con datos de estudiante
    resultado = promedio_est.merge(df_estudiantes, on='boleta', how='left')
    return resultado.sort_values('promedio_general', ascending=False).reset_index(drop=True)

## PARTE 3: Cálculos y Estadísticas

In [ ]:
def calcular_promedio_materia(df_calificaciones: pd.DataFrame, materia_id: str) -> Dict:
    """
    Calcula estadísticas de una materia específica.
    """
    df = df_calificaciones[df_calificaciones['materia_id'] == materia_id].copy()

    if df.empty:
        return {"materia": materia_id, "error": "No se encontraron registros"}

    df['promedio_alumno'] = df[['parcial_1', 'parcial_2', 'final']].mean(axis=1)

    resultado = {
        "materia": materia_id,
        "inscritos": len(df),
        "promedio_parcial1": round(df['parcial_1'].mean(), 2),
        "promedio_parcial2": round(df['parcial_2'].mean(), 2),
        "promedio_final": round(df['final'].mean(), 2),
        "promedio_general": round(df['promedio_alumno'].mean(), 2),
        "tasa_aprobacion": round((df['promedio_alumno'] >= 6.0).sum() / len(df) * 100, 1),
        "calificacion_maxima": df['promedio_alumno'].max(),
        "calificacion_minima": df['promedio_alumno'].min()
    }
    return resultado

In [ ]:
def ranking_estudiantes(df_calificaciones: pd.DataFrame,
                        df_estudiantes: pd.DataFrame,
                        top_n: int = 10) -> pd.DataFrame:
    """
    Genera ranking de mejores estudiantes por promedio general.
    """
    df = df_calificaciones.copy()
    df['promedio_materia'] = df[['parcial_1', 'parcial_2', 'final']].mean(axis=1)

    promedios = df.groupby('boleta')['promedio_materia'].mean().round(2).reset_index()
    promedios.columns = ['boleta', 'promedio_general']

    # Ordenar y tomar top N
    promedios = promedios.sort_values('promedio_general', ascending=False).head(top_n)
    promedios['posicion'] = range(1, len(promedios) + 1)

    # Unir con datos del estudiante
    resultado = promedios.merge(df_estudiantes[['boleta', 'nombre', 'semestre']], on='boleta', how='left')
    cols = ['posicion', 'boleta', 'nombre', 'semestre', 'promedio_general']
    return resultado[cols].reset_index(drop=True)

In [ ]:
def estadisticas_por_semestre(df_estudiantes: pd.DataFrame,
                              df_calificaciones: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula estadísticas agrupadas por semestre.
    """
    df = df_calificaciones.copy()
    df['promedio_materia'] = df[['parcial_1', 'parcial_2', 'final']].mean(axis=1)
    df['aprobada'] = df['promedio_materia'] >= 6.0

    # Promedio general por estudiante
    por_est = df.groupby('boleta').agg(
        promedio_general=('promedio_materia', 'mean'),
        tasa_aprobacion=('aprobada', 'mean')
    ).reset_index()

    # Unir con semestre
    por_est = por_est.merge(df_estudiantes[['boleta', 'semestre']], on='boleta', how='left')

    # Agrupar por semestre
    resultado = por_est.groupby('semestre').agg(
        num_estudiantes=('boleta', 'count'),
        promedio=('promedio_general', 'mean'),
        tasa_aprobacion=('tasa_aprobacion', 'mean'),
        mejor_promedio=('promedio_general', 'max'),
        peor_promedio=('promedio_general', 'min')
    ).round(2)

    resultado['tasa_aprobacion'] = (resultado['tasa_aprobacion'] * 100).round(1)
    resultado.columns = ['Estudiantes', 'Promedio', 'Tasa Aprob. (%)', 'Mejor Promedio', 'Peor Promedio']
    return resultado

## PARTE 4: Identificación de Riesgo y Reportes

In [ ]:
def identificar_estudiantes_riesgo(df_calificaciones: pd.DataFrame,
                                   df_estudiantes: pd.DataFrame,
                                   umbral_promedio: float = 7.0,
                                   max_reprobadas: int = 2) -> pd.DataFrame:
    """
    Identifica estudiantes en riesgo académico.
    Criterios: promedio < umbral O más de max_reprobadas materias reprobadas.
    """
    df = df_calificaciones.copy()
    df['promedio_materia'] = df[['parcial_1', 'parcial_2', 'final']].mean(axis=1)
    df['reprobada'] = df['promedio_materia'] < 6.0

    # Métricas por estudiante
    resumen = df.groupby('boleta').agg(
        promedio_general=('promedio_materia', 'mean'),
        materias_reprobadas=('reprobada', 'sum')
    ).reset_index()
    resumen['promedio_general'] = resumen['promedio_general'].round(2)
    resumen['materias_reprobadas'] = resumen['materias_reprobadas'].astype(int)

    # Evaluar criterios
    bajo_promedio = resumen['promedio_general'] < umbral_promedio
    muchas_reprobadas = resumen['materias_reprobadas'] > max_reprobadas

    en_riesgo = resumen[bajo_promedio | muchas_reprobadas].copy()

    # Asignar motivo
    def motivo(row):
        b = row['promedio_general'] < umbral_promedio
        r = row['materias_reprobadas'] > max_reprobadas
        if b and r:
            return 'Ambos'
        elif b:
            return 'Bajo promedio'
        else:
            return 'Mat. reprobadas'

    en_riesgo['motivo'] = en_riesgo.apply(motivo, axis=1)

    # Unir con nombre del estudiante
    en_riesgo = en_riesgo.merge(df_estudiantes[['boleta', 'nombre', 'semestre']], on='boleta', how='left')
    cols = ['boleta', 'nombre', 'semestre', 'promedio_general', 'materias_reprobadas', 'motivo']
    return en_riesgo[cols].sort_values('promedio_general').reset_index(drop=True)

In [ ]:
def generar_reporte_academico(df_estudiantes: pd.DataFrame,
                              df_calificaciones: pd.DataFrame,
                              df_materias: pd.DataFrame) -> Dict:
    """
    Genera reporte académico completo.
    """
    # Calcular promedio por materia y aprobación
    df = df_calificaciones.copy()
    df['promedio_materia'] = df[['parcial_1', 'parcial_2', 'final']].mean(axis=1)
    df['aprobada'] = df['promedio_materia'] >= 6.0

    promedio_global = round(df['promedio_materia'].mean(), 2)
    tasa_global = round(df['aprobada'].mean() * 100, 1)

    # Estadísticas por materia
    por_materia_agg = df.groupby('materia_id').agg(
        inscritos=('boleta', 'count'),
        promedio=('promedio_materia', 'mean'),
        tasa_aprobacion=('aprobada', 'mean')
    ).round(2).reset_index()
    por_materia_agg['tasa_aprobacion'] = (por_materia_agg['tasa_aprobacion'] * 100).round(1)
    por_materia_agg = por_materia_agg.merge(df_materias[['materia_id', 'nombre']], on='materia_id', how='left')

    reporte = {
        "resumen_general": {
            "total_estudiantes": len(df_estudiantes),
            "promedio_global": promedio_global,
            "tasa_aprobacion": tasa_global
        },
        "por_semestre": estadisticas_por_semestre(df_estudiantes, df_calificaciones),
        "por_materia": por_materia_agg,
        "mejores_estudiantes": ranking_estudiantes(df_calificaciones, df_estudiantes, top_n=5),
        "estudiantes_riesgo": identificar_estudiantes_riesgo(df_calificaciones, df_estudiantes),
        "fecha_generacion": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    return reporte

In [ ]:
def exportar_kardex(boleta: str, kardex: Dict, formato: str = 'csv') -> str:
    """
    Exporta el kardex de un estudiante a archivo CSV o JSON.
    Retorna: nombre del archivo generado.
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    nombre_archivo = f"kardex_{boleta}_{timestamp}.{formato}"

    if formato == 'csv':
        df_export = kardex['materias'].copy() if kardex['materias'] is not None else pd.DataFrame()
        # Agregar resumen al final
        resumen = pd.DataFrame([{
            'materia_id': 'RESUMEN',
            'nombre': '',
            'parcial_1': '',
            'parcial_2': '',
            'final': '',
            'promedio': kardex['promedio_general'],
            'estatus': f"Aprobadas: {kardex['materias_aprobadas']} | Reprobadas: {kardex['materias_reprobadas']}",
            'creditos': kardex['creditos_cursados']
        }])
        df_export = pd.concat([df_export, resumen], ignore_index=True)
        df_export.to_csv(nombre_archivo, index=False, encoding='utf-8-sig')

    elif formato == 'json':
        datos_json = {
            "estudiante": kardex['estudiante'],
            "promedio_general": kardex['promedio_general'],
            "creditos_cursados": kardex['creditos_cursados'],
            "materias_aprobadas": kardex['materias_aprobadas'],
            "materias_reprobadas": kardex['materias_reprobadas'],
            "materias": kardex['materias'].to_dict(orient='records') if kardex['materias'] is not None else []
        }
        with open(nombre_archivo, 'w', encoding='utf-8') as f:
            json.dump(datos_json, f, ensure_ascii=False, indent=2)
    else:
        raise ValueError(f"Formato '{formato}' no soportado. Usa 'csv' o 'json'.")

    print(f"✅ Kardex exportado: {nombre_archivo}")
    return nombre_archivo

## Funciones de Visualización

In [ ]:
def mostrar_kardex(kardex: Dict) -> None:
    """Muestra el kardex de forma legible."""
    if kardex['estudiante'] is None:
        print("❌ Estudiante no encontrado")
        return

    est = kardex['estudiante']
    print("=" * 70)
    print("                         KARDEX ACADÉMICO")
    print("=" * 70)
    print(f"\n📋 DATOS DEL ESTUDIANTE")
    print("-" * 40)
    print(f"Boleta:   {est.get('boleta', 'N/A')}")
    print(f"Nombre:   {est.get('nombre', 'N/A')}")
    print(f"Semestre: {est.get('semestre', 'N/A')}")
    print(f"Carrera:  {est.get('carrera', 'N/A')}")
    print(f"Email:    {est.get('email', 'N/A')}")

    print(f"\n📚 CALIFICACIONES")
    print("-" * 70)
    if kardex['materias'] is not None and len(kardex['materias']) > 0:
        print(kardex['materias'].to_string(index=False))
    else:
        print("Sin calificaciones registradas")

    print(f"\n📊 RESUMEN")
    print("-" * 40)
    print(f"Promedio General:    {kardex.get('promedio_general', 0):.2f}")
    print(f"Créditos Cursados:   {kardex.get('creditos_cursados', 0)}")
    print(f"Materias Aprobadas:  {kardex.get('materias_aprobadas', 0)}")
    print(f"Materias Reprobadas: {kardex.get('materias_reprobadas', 0)}")
    print("=" * 70)

In [ ]:
def mostrar_reporte(reporte: Dict) -> None:
    """Muestra el reporte académico completo."""
    print("=" * 70)
    print("              REPORTE ACADÉMICO - CIENCIA DE DATOS")
    print(f"              Generado: {reporte['fecha_generacion']}")
    print("=" * 70)

    res = reporte.get('resumen_general', {})
    print(f"\n📊 RESUMEN GENERAL")
    print("-" * 40)
    print(f"Total de estudiantes: {res.get('total_estudiantes', 'N/A')}")
    print(f"Promedio global:      {res.get('promedio_global', 0):.2f}")
    print(f"Tasa de aprobación:   {res.get('tasa_aprobacion', 0):.1f}%")

    if reporte.get('por_semestre') is not None:
        print(f"\n📅 ESTADÍSTICAS POR SEMESTRE")
        print("-" * 60)
        print(reporte['por_semestre'].to_string())

    if reporte.get('mejores_estudiantes') is not None:
        print(f"\n🏆 TOP 5 ESTUDIANTES")
        print("-" * 60)
        print(reporte['mejores_estudiantes'].to_string(index=False))

    if reporte.get('estudiantes_riesgo') is not None and len(reporte['estudiantes_riesgo']) > 0:
        print(f"\n⚠️  ESTUDIANTES EN RIESGO ({len(reporte['estudiantes_riesgo'])})")
        print("-" * 60)
        print(reporte['estudiantes_riesgo'].to_string(index=False))
    else:
        print(f"\n✅ No hay estudiantes en riesgo académico")

    print("\n" + "=" * 70)

## BONUS: Funcionalidades Extra

In [ ]:
def predecir_riesgo_proximo_semestre(df_calificaciones: pd.DataFrame,
                                     df_estudiantes: pd.DataFrame) -> pd.DataFrame:
    """
    Predice estudiantes que podrían estar en riesgo el próximo semestre
    basándose en tendencias decrecientes de calificaciones.
    Criterios:
      - Tendencia decreciente: parcial_2 < parcial_1
      - Calificación final menor que parcial_1
    """
    df = df_calificaciones.dropna(subset=['parcial_1', 'parcial_2', 'final']).copy()

    df['tendencia_decreciente'] = df['parcial_2'] < df['parcial_1']
    df['final_menor_p1'] = df['final'] < df['parcial_1']
    df['señal_riesgo'] = df['tendencia_decreciente'] & df['final_menor_p1']

    # Contar señales por estudiante
    resumen = df.groupby('boleta').agg(
        total_materias=('materia_id', 'count'),
        senales_riesgo=('señal_riesgo', 'sum')
    ).reset_index()

    resumen['pct_riesgo'] = (resumen['senales_riesgo'] / resumen['total_materias'] * 100).round(1)

    # Considera en riesgo futuro si >50% de materias muestran señal
    en_riesgo_futuro = resumen[resumen['pct_riesgo'] > 50].copy()
    en_riesgo_futuro = en_riesgo_futuro.merge(
        df_estudiantes[['boleta', 'nombre', 'semestre']], on='boleta', how='left'
    )
    cols = ['boleta', 'nombre', 'semestre', 'senales_riesgo', 'total_materias', 'pct_riesgo']
    return en_riesgo_futuro[cols].sort_values('pct_riesgo', ascending=False).reset_index(drop=True)

In [ ]:
def comparar_estudiantes(boleta1: str, boleta2: str,
                         df_calificaciones: pd.DataFrame,
                         df_estudiantes: pd.DataFrame,
                         df_materias: pd.DataFrame) -> Dict:
    """
    Compara el rendimiento de dos estudiantes.
    """
    k1 = obtener_kardex(boleta1, df_estudiantes, df_calificaciones, df_materias)
    k2 = obtener_kardex(boleta2, df_estudiantes, df_calificaciones, df_materias)

    # Materias en común
    if k1['materias'] is not None and k2['materias'] is not None:
        ids1 = set(k1['materias']['materia_id'])
        ids2 = set(k2['materias']['materia_id'])
        comunes = ids1 & ids2
        m1 = k1['materias'][k1['materias']['materia_id'].isin(comunes)][['materia_id', 'nombre', 'promedio']]
        m2 = k2['materias'][k2['materias']['materia_id'].isin(comunes)][['materia_id', 'promedio']]
        comparacion = m1.merge(m2, on='materia_id', suffixes=('_est1', '_est2'))
        comparacion['diferencia'] = (comparacion['promedio_est1'] - comparacion['promedio_est2']).round(2)
    else:
        comparacion = pd.DataFrame()

    return {
        "estudiante_1": k1['estudiante'],
        "estudiante_2": k2['estudiante'],
        "promedio_1": k1['promedio_general'],
        "promedio_2": k2['promedio_general'],
        "ganador": k1['estudiante'].get('nombre') if (k1['promedio_general'] or 0) >= (k2['promedio_general'] or 0) else k2['estudiante'].get('nombre'),
        "materias_comunes": comparacion
    }

## Prueba de la Implementación

In [ ]:
# Cargar datos
df_estudiantes, df_calificaciones, df_materias = cargar_datos()

print("DATOS CARGADOS")
print("=" * 50)
print(f"\nEstudiantes ({len(df_estudiantes)} registros):")
print(df_estudiantes.head())
print(f"\nCalificaciones ({len(df_calificaciones)} registros):")
print(df_calificaciones.head())
print(f"\nMaterias ({len(df_materias)} registros):")
print(df_materias)

In [ ]:
print("\nINFORMACIÓN GENERAL")
print("=" * 50)
info = info_general(df_estudiantes, df_calificaciones)
for k, v in info.items():
    print(f"  {k}: {v}")

In [ ]:
print("\nVALIDACIÓN DE DATOS")
print("=" * 50)
validacion = validar_datos(df_calificaciones)
for k, v in validacion.items():
    print(f"  {k}: {v}")

In [ ]:
print("\nBÚSQUEDA DE ESTUDIANTES")
print("=" * 50)

print("\n-- Buscar por nombre 'María' --")
print(buscar_estudiante(df_estudiantes, 'nombre', 'María'))

print("\n-- Buscar por semestre 3 --")
print(buscar_estudiante(df_estudiantes, 'semestre', '3'))

print("\n-- Buscar por boleta '2022630001' --")
print(buscar_estudiante(df_estudiantes, 'boleta', '2022630001'))

In [ ]:
print("\nKARDEX DE ESTUDIANTE")
kardex = obtener_kardex('2021630001', df_estudiantes, df_calificaciones, df_materias)
mostrar_kardex(kardex)

In [ ]:
print("\nFILTRAR POR RENDIMIENTO (promedio >= 8.0)")
print("=" * 50)
print(filtrar_por_rendimiento(df_calificaciones, df_estudiantes, min_promedio=8.0))

In [ ]:
print("\nESTADÍSTICAS DE MATERIA: MAT101")
print("=" * 50)
stats = calcular_promedio_materia(df_calificaciones, 'MAT101')
for k, v in stats.items():
    print(f"  {k}: {v}")

In [ ]:
print("\nRANKING TOP 5 ESTUDIANTES")
print("=" * 50)
print(ranking_estudiantes(df_calificaciones, df_estudiantes, top_n=5).to_string(index=False))

In [ ]:
print("\nESTADÍSTICAS POR SEMESTRE")
print("=" * 50)
print(estadisticas_por_semestre(df_estudiantes, df_calificaciones).to_string())

In [ ]:
print("\nESTUDIANTES EN RIESGO")
print("=" * 50)
riesgo = identificar_estudiantes_riesgo(df_calificaciones, df_estudiantes)
print(riesgo.to_string(index=False))

In [ ]:
# Reporte completo
reporte = generar_reporte_academico(df_estudiantes, df_calificaciones, df_materias)
mostrar_reporte(reporte)

In [ ]:
# Exportar kardex en ambos formatos
archivo_csv = exportar_kardex('2021630001', kardex, formato='csv')
archivo_json = exportar_kardex('2021630001', kardex, formato='json')
print(f"Archivos generados: {archivo_csv}, {archivo_json}")

In [ ]:
# BONUS: Predicción de riesgo futuro
print("\nBONUS: PREDICCIÓN DE RIESGO PRÓXIMO SEMESTRE")
print("=" * 50)
prediccion = predecir_riesgo_proximo_semestre(df_calificaciones, df_estudiantes)
if len(prediccion) > 0:
    print(prediccion.to_string(index=False))
else:
    print("✅ Ningún estudiante muestra tendencia de riesgo")

In [ ]:
# BONUS: Comparar dos estudiantes
print("\nBONUS: COMPARACIÓN DE ESTUDIANTES")
print("=" * 50)
comp = comparar_estudiantes('2021630001', '2021630002', df_calificaciones, df_estudiantes, df_materias)
print(f"Estudiante 1: {comp['estudiante_1']['nombre']} → Promedio: {comp['promedio_1']}")
print(f"Estudiante 2: {comp['estudiante_2']['nombre']} → Promedio: {comp['promedio_2']}")
print(f"Mejor rendimiento: {comp['ganador']}")
print("\nMaterias en común:")
print(comp['materias_comunes'].to_string(index=False))